In [1]:
# Problem 8: Market Basket Analysis

import pandas as pd
from collections import defaultdict
from itertools import combinations

# a. Load and preprocess
df = pd.read_csv("Groceries_dataset.csv")

transactions = df.groupby(
    ["Member_number", "Date"]
)["itemDescription"].apply(
    lambda x: set(x)
).tolist()

N = len(transactions)
min_support = 0.001
min_count = min_support * N

# b. Generate frequent itemsets
counts = defaultdict(int)

for t in transactions:
    for item in t:
        counts[frozenset([item])] += 1

frequent = {
    x: c for x, c in counts.items()
    if c >= min_count
}

all_frequent = dict(frequent)
k = 2

while frequent:
    candidates = set()
    items = list(frequent)

    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            union = items[i] | items[j]
            if len(union) == k:
                candidates.add(union)

    counts = defaultdict(int)

    for t in transactions:
        for c in candidates:
            if c.issubset(t):
                counts[c] += 1

    frequent = {
        x: c for x, c in counts.items()
        if c >= min_count
    }

    all_frequent.update(frequent)
    k += 1

# c. Generate association rules
rules = []

for itemset, count in all_frequent.items():
    if len(itemset) > 1:
        support = count / N
        items = list(itemset)

        for i in range(1, len(items)):
            for a in combinations(items, i):
                a = frozenset(a)
                c = itemset - a

                confidence = support / (all_frequent[a] / N)
                lift = confidence / (all_frequent[c] / N)

                if confidence >= 0.05:
                    rules.append([
                        list(a), list(c), support,
                        confidence, lift
                    ])

rules = pd.DataFrame(
    rules,
    columns=["Antecedent", "Consequent",
             "Support", "Confidence", "Lift"]
)

print("\nTop Association Rules:")
print(rules.sort_values("Lift", ascending=False).head(10))


Top Association Rules:
                Antecedent         Consequent   Support  Confidence      Lift
424   [yogurt, whole milk]          [sausage]  0.001470    0.131737  2.182917
425  [sausage, whole milk]           [yogurt]  0.001470    0.164179  1.911760
239  [specialty chocolate]     [citrus fruit]  0.001403    0.087866  1.653762
423      [yogurt, sausage]       [whole milk]  0.001470    0.255814  1.619866
395                [flour]   [tropical fruit]  0.001069    0.109589  1.617141
211            [beverages]          [sausage]  0.001537    0.092742  1.536764
448     [soda, whole milk]          [sausage]  0.001069    0.091954  1.523708
216              [napkins]           [pastry]  0.001738    0.078550  1.518529
414     [processed cheese]  [root vegetables]  0.001069    0.105263  1.513019
381          [hard cheese]        [pip fruit]  0.001069    0.072727  1.482586
